<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第4章：通过推理时扩展提升推理能力

本 Notebook 使用到的依赖包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "matplotlib",
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F01_raschka.webp" width=600>

&nbsp;
## 4.1 推理时扩展简介

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F02_raschka.webp" width=600>

- 上图灵感来自 OpenAI o1 博文 https://openai.com/index/learning-to-reason-with-llms/

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F03_raschka.webp" width=600>

- 这三种方法都能把基础模型在 MATH-500（上一章的基准）上的准确率提升到两倍以上。
- 第 1、2 种方法在本章讲解，第 3 种方法放到下一章。

&nbsp;
## 4.2 加载预训练模型

- 模型加载代码与第 2、3 章类似。

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()

# 本章首次运行时使用 CPU
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 先用上一章接触过的 MATH-500 数据集题目来试试模型：

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

print(prompt)

- 下方这个函数基于上一章的 `generate_text_stream_concat`，不过我们稍作调整，以便后续可以传入其他 `generate_*` 函数。

In [ ]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def generate_text_stream_concat_flex(
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False, 
    generate_func=None,  # 新增
    **generate_kwargs  # 新增
):

    if generate_func is None:  # 新增
        generate_func = generate_text_basic_stream_cache
        
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
        ).unsqueeze(0)

    generated_ids = []
    for token in generate_func(  # 新增
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        **generate_kwargs,  # 新增
    ):
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())

        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )
    return tokenizer.decode(generated_ids)

In [ ]:
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_basic_stream_cache  # 新增
)

- 顺带一提，正确答案是 83。

&nbsp;
## 4.3 使用链式思维提示来生成更好的回答

- 提升模型表现的经典方法之一，就是改写提示语，鼓励它逐步思考或逐步解释。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F04_raschka.webp" width=600>

In [ ]:
prompt_cot = prompt + " \n\nExplain step by step."

response_cot = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
)

&nbsp;
## 4.4 通过温度缩放控制输出多样性

- 到目前为止我们使用的都是确定性解码，本节将引入随机解码。
- 先从温度缩放开始，随后在第 4.5 节加入 top-p，以更好地平衡随机性与连贯性。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F05_raschka.webp" width=600>

&nbsp;
### 4.4.1 理解选择下一个 token 的过程

In [ ]:
ex_prompt = "The capital of Germany is"

response = generate_text_stream_concat_flex(
    model, tokenizer, ex_prompt, device,
    max_new_tokens=1, verbose=True
)

- 内部具体发生了什么：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F06_raschka.webp" width=600>

In [ ]:
# 1）将输入文本转换为词元 ID

input_token_ids = torch.tensor(
    tokenizer.encode(ex_prompt), device=device
).unsqueeze(0)

print(input_token_ids)

In [ ]:
# 2）获取下一个词元的分数

with torch.inference_mode():
    next_token_logits = model(input_token_ids)[:, -1]
print(next_token_logits.shape)  # 形状：[1, 词表大小]

In [ ]:
# 3）查找分数最高的词表索引

max_token_id = torch.argmax(next_token_logits)
print(f"Token ID: {max_token_id}")
print(f"Decoded token: '{tokenizer.decode([max_token_id])}'")

- 该 LLM 的词表包含 151,936 个词元，词元索引范围为 0 到 151,935；为了便于可视化，这里只关注索引 19,800 到 19,900 之间的 100 个词元。


In [ ]:
import matplotlib.pyplot as plt

def plot_scores_bar(
    next_token_logits, start=19_800, end=19_900,
    arrow=True, ylabel="Logit value"
):
    # 选择词表子区间
    x = torch.arange(start, end)

    # .cpu() 是 to(torch.device("cpu")) 的简写
    logits_section = next_token_logits[0, start:end].float().cpu()

    # 绘制 logits
    plt.bar(x, logits_section)
    plt.xlabel("Vocabulary index")
    plt.ylabel(ylabel)

    # 突出显示最大 logit
    if arrow:
        max_idx = torch.argmax(logits_section)
        plt.annotate(
            "Berlin",
            xy=(x[max_idx], logits_section[max_idx]),
            xytext=(x[max_idx] - 25, logits_section[max_idx] - 2),
            arrowprops={
                "facecolor": "black", "arrowstyle": "->", "lw": 1.5
            },
            fontsize=10,
        )

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_scores_bar(next_token_logits)

&nbsp;
### 4.4.2 使用温度参数重新缩放词元分数（logits）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F08_raschka.webp" width=600>

In [ ]:
def scale_logits_by_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    return logits / temperature

- 从严谨的可视化角度来说，柱状图更合适；不过折线图能更直观地对比差异。

In [ ]:
def plot_logits_with_temperature(
    next_token_logits, start=19_800, end=19_900,
    temps=(0.5, 5.0),
):
    x = torch.arange(start, end)
    logits_orig = next_token_logits[0, start:end].float().cpu()

    # 应用温度缩放
    logits_scaled = [
        scale_logits_by_temperature(logits_orig, T) for T in temps
    ]
    # 绘制 logits
    plt.plot(x, logits_orig, label="Original logits", lw=2)
    plt.plot(
        x, logits_scaled[0],
        label=f"T={temps[0]} (sharper)", ls="--", lw=1
    )
    plt.plot(
        x, logits_scaled[1],
        label=f"T={temps[1]} (flatter)", ls=":", lw=3
    )

    # 突出显示最大 logit
    max_idx = torch.argmax(logits_orig)
    plt.annotate(
        "Berlin",
        xy=(x[max_idx], logits_orig[max_idx]),
        xytext=(x[max_idx] - 25, logits_orig[max_idx] + 2),
        arrowprops={"facecolor": "black", "arrowstyle": "->", "lw": 1.5},
        fontsize=12,
    )

    plt.xlabel("Vocabulary index")
    plt.ylabel("Logit value")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_logits_with_temperature(
    next_token_logits,
    temps=(0.5, 5.0)
)

&nbsp;
### 4.4.3 从概率分布中采样下一个 token

- 将 logits 得分转化为概率得分（总和为 1）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F10_raschka.webp" width=600>

In [ ]:
# 第 3.2 步：重新缩放下一词元分数
rescaled_logits = scale_logits_by_temperature(next_token_logits, 5.0)

# 第 3.3 步：将重新缩放的 logits 转换为概率分数
next_token_probas = torch.softmax(
    rescaled_logits, dim=-1
)

print("Probability sum:", torch.sum(next_token_probas))

In [ ]:
plot_scores_bar(
    next_token_probas, arrow=False, ylabel="Probability value"
)

In [ ]:
print("Token ID 19,846 probability:", next_token_probas[:, 19846])

In [ ]:
print("Highest probability:", max(next_token_probas.squeeze(0)))

In [ ]:
# 第 3.4 步：按概率采样词元
torch.manual_seed(123)
print(
    "Sampled token:",
    torch.multinomial(next_token_probas.cpu(), num_samples=1)
)

In [ ]:
print(tokenizer.decode([65094]))

In [ ]:
def count_samples(probas, num_samples=1000, threshold=1, tokenizer=None):
    # 按概率抽取样本
    samples = torch.multinomial(
        probas.cpu(), num_samples=num_samples, replacement=True
    )
    
    # 统计每个索引被选中的次数
    counts = torch.bincount(samples.squeeze(0), minlength=1)
    
    # 输出结果
    for i, c in enumerate(counts):
        if c > threshold:
            if tokenizer is None:
                print(f"Vocab index {i}: {c.item()}x")
            else:
                print(f"'{tokenizer.decode([i])}': {c.item()}x")

In [ ]:
torch.manual_seed(123)
count_samples(next_token_probas, tokenizer=tokenizer)

In [ ]:
torch.manual_seed(123)
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
count_samples(probas_lowT, threshold=1, tokenizer=tokenizer)

In [ ]:
print(probas_lowT[0, 19_846])

- 之所以出现 `"______"`，很可能是模型在训练数据里见过类似格式，于是尝试为用户生成一道填空题。

&nbsp;
### 4.4.4 在文本生成函数中加入温度缩放参数

In [ ]:
# 附录 B 练习 2.2
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_temp_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=0.
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # 第 3.1 步：获取 logits
    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):

        ########################################
        # 新增:
        orig_device = token_ids.device

        if temperature is None or temperature == 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)

        else:
            # 第 3.2 步：对 logits 应用温度缩放
            logits = scale_logits_by_temperature(out, temperature)

            # 第 3.3 步：转换为概率
            probas = torch.softmax(logits, dim=-1)

            # 第 3.4 步：按概率采样词元
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)
            
        #########################################
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

In [ ]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_temp_stream_cache,
    temperature=1.1
)

&nbsp;
## 4.5 通过 top-p 采样平衡多样性与连贯性

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F12_raschka.webp" width=600>

&nbsp;
### 4.5.1 选择 top-p 子集 token

In [ ]:
# 第 3.1 步：获取 logits（这里对 10 个词元使用玩具 logits）
toy_logits = torch.tensor(
    [-0.7, -3.0, 0.1, -1.2, 2.0, -1.0, -0.5, -2.0, 0.3, 1.5]
)

# 第 3.2 步：应用温度缩放
toy_logits_scaled = scale_logits_by_temperature(toy_logits, 1.0)

# 第 3.3 步：转换为概率
toy_probas = torch.softmax(toy_logits_scaled, dim=-1)

plt.bar(
    torch.arange(len(toy_logits_scaled)), toy_probas,
    alpha=0.5
)

plt.ylim([0, 1])
plt.xlabel("Vocabulary index")
plt.ylabel("Probability")
# plt.savefig("12.pdf")
plt.show()

In [ ]:
# 第 4.1 步：按概率降序排序
sorted_probas, sorted_idx = torch.sort(toy_probas, descending=True)

# 第 4.2 步：计算累积和
cumsum = torch.cumsum(sorted_probas, dim=-1)

plt.bar(
    torch.arange(len(sorted_probas)), sorted_probas, 
        alpha=0.5
)
plt.step(
    torch.arange(len(cumsum)), cumsum, 
    where="mid", color="C1", label="Cumulative sum"
)

plt.ylim([0, 1])
plt.xlabel("Token rank (sorted by probability)")
plt.ylabel("Probability")
plt.show()

In [ ]:
# 第 4.3.1 步：应用 top-p 阈值（例如保留词元直到累积概率质量 > 0.8）
top_p = 0.8
keep_mask = cumsum <= top_p
n_kept = torch.sum(keep_mask).item()
print("Cumulative sum:", cumsum)
print("Tokens kept:", n_kept)

In [ ]:
# 更常见的变体会保留使累积值越过阈值的词元
keep_mask = (cumsum - sorted_probas) < top_p
n_kept = keep_mask.sum().item()
print("Tokens kept:", n_kept)

In [ ]:
plt.bar(
    torch.arange(len(sorted_probas)), sorted_probas, 
    alpha=0.5, label="Sorted probabilities"
)
plt.step(
    torch.arange(len(cumsum)), cumsum, where="mid",
    color="darkorange", label="Cumulative sum"
)

# 突出显示截断点
plt.axhline(
    top_p, color="red", linestyle="--",
    label=f"top_p = {top_p}"
)
plt.axvline(
    n_kept - 0.5, color="gray", linestyle=":",
    label=f"Top-p cutoff at {n_kept} tokens"
)

plt.xlabel("Token rank (sorted by probability)")
plt.ylabel("Probability")
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 1.05)
# plt.savefig("14.pdf")
plt.show()

In [ ]:
# 第 4.3.2 步：将截断点之后的值置零
kept_sorted = torch.where(
    keep_mask, sorted_probas,
    torch.zeros_like(sorted_probas)
)

# 第 4.3.3 步：映射回原始顺序
filtered = torch.zeros_like(toy_probas).scatter(0, sorted_idx, kept_sorted)

print(filtered)

In [ ]:
# 第 4.4 步：重新归一化，使总和为 1
denom = torch.sum(filtered).clamp_min(1e-12)
renormalized = filtered / denom
print(renormalized)

&nbsp;
### 4.5.2 在文本生成函数中加入 top-p 过滤器

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F16_raschka.webp" width=600>

In [ ]:
def top_p_filter(probas, top_p):
    if top_p is None or top_p >= 1.0:
        return probas

    # 第 4.1 步：按概率降序排序
    sorted_probas, sorted_idx = torch.sort(probas, dim=1, descending=True)

    # 第 4.2 步：计算累积和
    cumprobas = torch.cumsum(sorted_probas, dim=1)

    # 第 4.3.1 步：保留前缀累积概率质量（当前词元之前）小于 top_p 的词元
    # 示例：[0.5, 0.41, 0.09] 在 top_p=0.9 时应保留前两个词元
    prefix = cumprobas - sorted_probas   # 每个词元之前的累积概率质量
    keep = prefix < top_p
    # 始终至少保留一个词元（用于 top_p 很小或非正数时的回退）
    keep[:, 0] = True    

    # 第 4.3.2 步：将截断点之后的值置零
    kept_sorted = torch.where(
        keep, sorted_probas,
        torch.zeros_like(sorted_probas)
    )
    # 第 4.3.3 步：映射回原始顺序
    filtered = torch.zeros_like(probas).scatter(1, sorted_idx, kept_sorted)

    # 第 4.4 步：重新归一化，使总和为 1
    denom = torch.sum(filtered, dim=1, keepdim=True).clamp_min(1e-12)
    return filtered / denom

In [ ]:
with torch.inference_mode():
    next_token_logits = model(input_token_ids)[:, -1]
print(next_token_logits.shape)

In [ ]:
torch.manual_seed(123)
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
count_samples(probas_lowT, threshold=1, tokenizer=tokenizer)

In [ ]:
torch.manual_seed(123)
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
probas_lowT_filtered = top_p_filter(probas_lowT, top_p=0.8)

count_samples(probas_lowT_filtered, threshold=1, tokenizer=tokenizer)

In [ ]:
@torch.inference_mode()
def generate_text_top_p_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=0.,
    top_p=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # 第 3.1 步：获取 logits
    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):

        orig_device = token_ids.device

        if temperature is None or temperature == 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)

        else:
            # 第 3.2 步：对 logits 应用温度缩放
            logits = scale_logits_by_temperature(out, temperature)

            # 第 3.3 步：转换为概率
            probas = torch.softmax(logits, dim=-1)

            # （新增）第 4 步：对概率应用 top-p 过滤
            probas = top_p_filter(probas, top_p)            

            # 第 3.4 步：按概率采样词元
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

In [ ]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.5,
    top_p=0.8, 
)

&nbsp;
## 4.6 通过自洽性提升回答准确率

- 通过采样生成多个答案（或推理链），再进行投票。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F17_raschka.webp" width=600>

- 原理类似传统的多数投票，但这里不是多个模型投票，而是通过不同的温度设置来采样。
- 该方法也出现在 [Self-Consistency Improves Chain-of-Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171) paper
- 简言之，具体流程如下：
  - 在较高温度和 top-p 设置下采样多条推理链；
  - 从每条推理链中提取最终答案；
  - 选取出现次数最多的那个答案。
- 注：严格讲这在英语里称为 [plurality vote](https://en.wikipedia.org/wiki/Plurality_(voting))（相对多数投票），和“majority vote”并不完全相同。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F18_raschka.webp?1" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate
from collections import Counter

def self_consistency_vote(
    model, tokenizer, prompt, device,
    num_samples=10, temperature=0.8, top_p=0.9, max_new_tokens=2048,
    show_progress=True, show_long_answer=False, seed=None,
):
    full_answers, short_answers = [], []

    # 1）采样多个答案
    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model, tokenizer=tokenizer, prompt=prompt, device=device,
            max_new_tokens=max_new_tokens, verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature, top_p=top_p,
        )

        # 2）从每个答案中提取最终（简短）答案
        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    # 3）选择出现频率最高的最终答案（自洽投票）
    counts = Counter(short_answers)
    groups = {s: [] for s in counts}
    for idx, s in enumerate(short_answers):
        groups[s].append(idx)

    mc = counts.most_common()
    if not mc:
        majority_winners, final_answer = [], None
    else:
        top_freq = mc[0][1]
        majority_winners = [s for s, f in mc if f == top_freq]
        final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

- 额外建议：
  - 如果多个长答案几乎一模一样，可以逐步调高温度，增加差异性；
  - 如果答案整体偏离题意或质量较差，就调低温度，约束生成。

In [ ]:
results = self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

In [ ]:
print(results["final_answer"])

In [ ]:
print(results["full_answers"][0])

In [ ]:
results = self_consistency_vote(
    model,
    tokenizer,
    prompt + "\n\nExplain step by step.",
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

- Claude 4 也采用了类似的并行采样方式（https://www.anthropic.com/news/claude-4）；此外，Claude 4 还使用内部模型为回复评分
- 评分器可以类似于附录 F（F.5 使用其他 LLM 评判回复）中的 LLM 评判器
- 下一章还会介绍一种基于置信度分数的评分方法（也可以用它来打破平局）


下表展示了不同方法在 CUDA 设备（DGX Spark）上评测完整 MATH-500 测试集（500 题）的结果。

|    | 方法                                       | 模型     | 准确率 | 耗时       |
|----|----------------------------------------------|-----------|----------|------------|
| 1  | 基线（第 3 章），贪婪解码        | 基础      | 15.2%    | 10.1 分钟   |
| 2  | 基线（第 3 章），贪婪解码        | 推理 | 48.2%    | 182.1 分钟  |
| 3  | 思维链提示（"CoT"）           | 基础      | 40.6%    | 84.5 分钟   |
| 4  | 温度与 top-p（"Top-p"）              | 基础      | 17.8%    | 30.7 分钟   |
| 5  | "Top-p" + 自洽性 (n=3)             | 基础      | 29.6%    | 97.6 分钟   |
| 6  | "Top-p" + 自洽性 (n=5)             | 基础      | 27.8%    | 116.8 分钟  |
| 7  | "Top-p" + 自洽性 (n=10)            | 基础      | 31.6%    | 300.4 分钟  |
| 8  | "Top-p" + "CoT"                              | 基础      | 33.4%    | 129.2 分钟  |
| 9  | 自洽性 (n=3) + "Top-p" + "CoT"     | 基础      | 42.2%    | 211.6 分钟  |
| 10 | 自洽性 (n=5) + "Top-p" + "CoT"     | 基础      | 48.0%    | 452.9 分钟  |
| 11 | 自洽性 (n=10) + "Top-p" + "CoT"    | 基础      | 52.0%    | 862.6 分钟  |
| 12 | 自洽性 (n=3) + "Top-p" + "CoT"     | 推理 | 55.2%    | 544.4 分钟  |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F19_raschka.webp" width=600>

&nbsp;
## 4.7  总结

- 本节没有代码。